#### **Draft - Simple MLP and masks**
##### 20 April 2026

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
import pickle
import pandas as pd
import numpy as np

### Implementation of a simple MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size=14*14, interm_size_1=64, interm_size_2=128, interm_size_3=256, num_classes=10):
        super(MLP, self).__init__()

        self.fc1 = nn.Linear(input_size, interm_size_1)
        self.fc2 = nn.Linear(interm_size_1, interm_size_2)
        self.fc3 = nn.Linear(interm_size_2, interm_size_3)
        self.fc4 = nn.Linear(interm_size_3, num_classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten input
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
model= MLP()
for name, module in model.named_modules():
    print(name, module)

 MLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=256, bias=True)
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)
fc1 Linear(in_features=196, out_features=64, bias=True)
fc2 Linear(in_features=64, out_features=128, bias=True)
fc3 Linear(in_features=128, out_features=256, bias=True)
fc4 Linear(in_features=256, out_features=10, bias=True)


### Implementation of a "MaskLayer"

In [ ]:
class MaskLayer(nn.Module):
    def __init__(self, mask_params, probs=False):
        super(MaskLayer, self).__init__()

        self.probs = probs

        if self.probs and (torch.any(mask_params > 1) or torch.any(mask_params < 0)):
            assert False

        self.mask_params = nn.Parameter(mask_params, requires_grad=True)

    def forward(self, x):

        if self.probs:
            # boolean_mask_params = ((self.mask_params >= 0.5).float() - self.mask_params).detach() + self.mask_params    # "detach trick" for straight-through estimator
            boolean_mask_params = (self.mask_params >= 0.5).float()
        else:
            boolean_mask_params = self.mask_params

        masked_output = x * boolean_mask_params     # The output of each neuron in the masked layer (after activations)
                                                    # is multiplied by the corresponding (boolean) element of the mask.

        return masked_output

**Note:** we implement the masks in such a way to reproduce the behavior of "*structured pruning*". It means that each element in a mask corresponds to a neuron in the original layer that is being masked.

### Implementation of a masked version of the MLP model

In [ ]:
class MaskedMLP(nn.Module):
    def __init__(self, model, dict_mask_params, probs=False):
        super(MaskedMLP, self).__init__()

        # Note: 'probs' = True means that the mask params in dict_mask_params are between 0 and 1.
        #       'probs' = False means that the mask params are boolean (0 or 1).

        self.fc1 = copy.deepcopy(model.fc1)
        self.mask1 = MaskLayer(dict_mask_params['fc1'], probs)

        self.fc2 = copy.deepcopy(model.fc2)
        self.mask2 = MaskLayer(dict_mask_params['fc2'], probs)

        self.fc3 = copy.deepcopy(model.fc3)
        self.mask3 = MaskLayer(dict_mask_params['fc3'], probs)

        self.fc4 = copy.deepcopy(model.fc4) # We don't apply a mask to fc4, as we don't want to prune entire neurons in the last layer 

        # Saving mask layers in a list
        self.mask_layers = [self.mask1, self.mask2, self.mask3]


    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten input

        x = F.relu(self.fc1(x))
        x = self.mask1(x)           # In our implementation, we are applying the mask after the activations

        x = F.relu(self.fc2(x))
        x = self.mask2(x)

        x = F.relu(self.fc3(x))
        x = self.mask3(x)

        x = self.fc4(x)

        return x

#### Instantiating an MLP model (and saving it, without masks)

In [ ]:
model = MLP()
torch.save(model.state_dict(), "mlp_model.pth")

model

MLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=256, bias=True)
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)

#### Creating the masks

In [ ]:
def generate_random_vector(N, p):
    # (You can ignore the implementation details here. This function is just used to generate random masks, in our example.)
    # This function generates a vector of length N, with random values between 0 and 1,
    # and such that a percentage p (0 <= p <= 1) of its elements are smaller than 0.5.

    m = torch.rand(N) < p

    x = torch.zeros(N)
    x[m] = torch.rand(m.sum()) * 0.5
    x[~m] = 0.5 + torch.rand((~m).sum()) * 0.5

    return x



# Let's suppose that we want:
#   percentage of pruned neurons in layer fc1: 20%
#   percentage of pruned neurons in layer fc2: 40%
#   percentage of pruned neurons in layer fc3: 30%

dict_mask_params_probs = {
    'fc1': generate_random_vector(N=model.fc1.out_features, p=0).requires_grad_(True),
    'fc2': generate_random_vector(N=model.fc2.out_features, p=0.8).requires_grad_(True),
    'fc3': generate_random_vector(N=model.fc3.out_features, p=0.6).requires_grad_(True),
    'fc4': generate_random_vector(N=model.fc4.out_features, p=0).requires_grad_(True)
}

# Boolean ("binarized") version of the masks
dict_mask_params_boolean = {k:(v >= 0.5).float() for k,v in dict_mask_params_probs.items()}


# Saving the "probability" and "boolean" versions of the masks
with open('dict_mask_params_probs.pkl', 'wb') as f:
    pickle.dump(dict_mask_params_probs, f)
with open('dict_mask_params_boolean.pkl', 'wb') as f:
    pickle.dump(dict_mask_params_boolean, f)

In [ ]:
dict_mask_params_probs

{'fc1': tensor([0.6636, 0.7120, 0.6713, 0.6007, 0.7534, 0.5191, 0.6561, 0.8144, 0.9298,
         0.7073, 0.6018, 0.9083, 0.6307, 0.8351, 0.7977, 0.5844, 0.9712, 0.7262,
         0.6206, 0.7965, 0.9201, 0.7331, 0.9007, 0.8940, 0.8543, 0.9287, 0.8057,
         0.5543, 0.8975, 0.8987, 0.6057, 0.8755, 0.6769, 0.6818, 0.5789, 0.8143,
         0.7060, 0.6165, 0.7252, 0.9854, 0.8418, 0.5857, 0.8162, 0.5177, 0.7287,
         0.8688, 0.5873, 0.6603, 0.6306, 0.8243, 0.5854, 0.8573, 0.6109, 0.9485,
         0.6717, 0.6198, 0.5595, 0.5163, 0.7931, 0.9044, 0.7717, 0.6341, 0.9927,
         0.5784], requires_grad=True),
 'fc2': tensor([2.9390e-01, 5.2710e-01, 4.2107e-01, 1.7795e-01, 1.6727e-01, 4.9051e-01,
         3.0246e-01, 1.3254e-01, 7.1467e-02, 3.7342e-02, 3.5672e-01, 4.5875e-01,
         7.3232e-01, 7.4877e-02, 2.2364e-01, 2.6288e-02, 5.2077e-01, 3.1980e-01,
         1.1290e-01, 1.0307e-01, 4.9538e-02, 8.3820e-02, 3.9804e-01, 2.7478e-02,
         8.1976e-02, 2.6978e-01, 1.9262e-02, 2.8776e-01,

#### Instantiating a masked version of our MLP, using the random masks generated:

In [ ]:
dict_mask_params_probs

{'fc1': tensor([0.6636, 0.7120, 0.6713, 0.6007, 0.7534, 0.5191, 0.6561, 0.8144, 0.9298,
         0.7073, 0.6018, 0.9083, 0.6307, 0.8351, 0.7977, 0.5844, 0.9712, 0.7262,
         0.6206, 0.7965, 0.9201, 0.7331, 0.9007, 0.8940, 0.8543, 0.9287, 0.8057,
         0.5543, 0.8975, 0.8987, 0.6057, 0.8755, 0.6769, 0.6818, 0.5789, 0.8143,
         0.7060, 0.6165, 0.7252, 0.9854, 0.8418, 0.5857, 0.8162, 0.5177, 0.7287,
         0.8688, 0.5873, 0.6603, 0.6306, 0.8243, 0.5854, 0.8573, 0.6109, 0.9485,
         0.6717, 0.6198, 0.5595, 0.5163, 0.7931, 0.9044, 0.7717, 0.6341, 0.9927,
         0.5784], requires_grad=True),
 'fc2': tensor([2.9390e-01, 5.2710e-01, 4.2107e-01, 1.7795e-01, 1.6727e-01, 4.9051e-01,
         3.0246e-01, 1.3254e-01, 7.1467e-02, 3.7342e-02, 3.5672e-01, 4.5875e-01,
         7.3232e-01, 7.4877e-02, 2.2364e-01, 2.6288e-02, 5.2077e-01, 3.1980e-01,
         1.1290e-01, 1.0307e-01, 4.9538e-02, 8.3820e-02, 3.9804e-01, 2.7478e-02,
         8.1976e-02, 2.6978e-01, 1.9262e-02, 2.8776e-01,

In [ ]:
masked_model = MaskedMLP(model, dict_mask_params_probs, probs=True)

masked_model

MaskedMLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (mask1): MaskLayer()
  (fc2): Linear(in_features=64, out_features=128, bias=True)
  (mask2): MaskLayer()
  (fc3): Linear(in_features=128, out_features=256, bias=True)
  (mask3): MaskLayer()
  (fc4): Linear(in_features=256, out_features=10, bias=True)
)

# Test Function 

In [ ]:
from energy_estimator import estimate_model_energy

In [ ]:
result = estimate_model_energy(
    model=model,
    board="JetsonNano",
    masks=dict_mask_params_probs,
)

Layer fc1 is masked. Effective output features: 47.258758544921875
lo: 1, hi: 2, t: 0.53125
lo: 0, hi: 1, t: 0.0
Layer fc2 is masked. Effective output features: 41.68940353393555
lo: 0, hi: 1, t: 0.0
lo: 0, hi: 1, t: 0.0
Layer fc3 is masked. Effective output features: 112.01551818847656
lo: 0, hi: 1, t: 1.0
lo: 0, hi: 1, t: 0.7502424716949463
Layer fc4 is masked. Effective output features: 6.985209941864014
lo: 1, hi: 2, t: 1.0
lo: 0, hi: 1, t: 0.0


c:\Users\AissaABDELAZIZ\Desktop\package python\energy_estimator\model_energy_aggregation.py:62: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  "out": float(eff_out) if not isinstance(eff_out, torch.Tensor) else float(eff_out),


In [ ]:
result["total_energy"]=result["total_energy"]* 1000000

In [ ]:
result["total_energy"].backward()

In [ ]:
dict_mask_params_probs["fc3"].grad

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])